<a href="https://colab.research.google.com/github/kihahu/kikuyu-tts/blob/initial-import/notebooks/train_kikuyu_vits_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Train Kikuyu VITS From Scratch (WaxalNLP `kik_tts`)

This notebook runs the full pipeline in Colab:
1. Environment setup (Coqui TTS from PyPI; optional separate clone in **Advanced**)
2. Dataset prep + `train_coqui.txt` / `dev_coqui.txt` (Coqui formatter)
3. Char tokenizer
4. VITS from scratch via `scripts/colab_train_vits_scratch.py` (resume + Drive in config)
5. Optional HF Hub push, evaluation, local integration bundle


In [ ]:
!pip install -U pip setuptools wheel
!pip install 'datasets[audio]' soundfile librosa pyyaml huggingface_hub
!pip install coqui-tts


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!cd /content && ([ -d kikuyu-tts ] || git clone https://github.com/kihahu/kikuyu-tts.git) && cd kikuyu-tts && git pull
%cd /content/kikuyu-tts


In [ ]:
!python scripts/prepare_waxal_kik_tts.py \
  --dataset-name google/WaxalNLP \
  --dataset-config kik_tts \
  --split train \
  --output-dir data/waxal_kik_tts \
  --target-sample-rate 16000 \
  --min-duration-sec 0.6 \
  --max-duration-sec 25.0 \
  --min-rms 0.0035 \
  --seed 42 \
  --dev-ratio 0.10 \
  --test-ratio 0.05


In [ ]:
!python scripts/build_kikuyu_vocab.py \
  --train-manifest data/waxal_kik_tts/manifests/train.jsonl \
  --dev-manifest data/waxal_kik_tts/manifests/dev.jsonl \
  --out-dir artifacts/tokenizer_kikuyu_char


In [ ]:
# Optional: if you have broken `TTS`/`trainer` wheel state, nuke the cache and reinstall.
# !pip uninstall -y TTS trainer coqpit
# !pip cache purge
# !pip install -U coqui-tts
# !pip install -e .
# %cd /content/kikuyu-tts


In [ ]:
# Start fresh (writes coqui_vits_config.yaml, then runs train_tts)
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts


In [ ]:
# Resume from latest checkpoint on Drive (see local_output_path / config)
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts \
  --resume


In [ ]:
# Optional: push checkpoints to Hugging Face Hub
!huggingface-cli login
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts \
  --resume \
  --push-hf


Create a metrics CSV at `artifacts/checkpoint_metrics.csv` with columns:
- `checkpoint`
- `synthesis_success_rate`
- `clipping_rate`
- `mos_lite`
- `wer_proxy`


In [ ]:
!python scripts/evaluate_and_select.py \
  --metrics-csv artifacts/checkpoint_metrics.csv \
  --out-json artifacts/best_checkpoint_selection.json \
  --out-csv artifacts/tts_eval_summary.csv


In [ ]:
!python scripts/prepare_local_integration.py \
  --best-checkpoint-dir artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best \
  --tokenizer-dir artifacts/tokenizer_kikuyu_char \
  --eval-summary-csv artifacts/tts_eval_summary.csv \
  --out-dir artifacts/local_integration/kikuyu_vits_best \
  --model-id kikuyu-vits-scratch-waxal
